# 05 - Generate Insert SQL

Generate data load script `04_load_data.sql` dynamically based on the cleaned CSVs and mappings.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Paths
ROOT = Path().resolve().parent
CLEANED_DIR = ROOT / "data" / "cleaned"
MAPPINGS_DIR = ROOT / "reports" / "key_mappings"
SQL_OUT = ROOT / "sql" / "04_load_data.sql"

In [3]:
def escape_sql(val):
    if pd.isna(val):
        return 'NULL'
    if isinstance(val, (int, np.integer)):
        return str(val)
    if isinstance(val, (float, np.floating)):
        return str(val)
    if isinstance(val, bool):
        return '1' if val else '0'
    s = str(val).replace("'", "''")
    return f"N'{s}'"

def generate_inserts(df, table_name, columns, batch_size=1000):
    lines = []
    df = df.replace({np.nan: None})
    
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start+batch_size]
        lines.append(f"INSERT INTO dbo.{table_name} ({', '.join(columns)}) VALUES")
        values_list = []
        for _, row in batch.iterrows():
            vals = [escape_sql(row[col]) for col in columns]
            values_list.append(f"({', '.join(vals)})")
        lines.append(",\n".join(values_list) + ";")
    lines.append("GO\n")
    return "\n".join(lines)

def trim_cols(df):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()
    return df


In [4]:
f = open(SQL_OUT, "w", encoding="utf-8")
f.write("USE MedicineDB;\nGO\n\n")

20

In [5]:
# 1. Manufacturers
mfr = pd.read_csv(CLEANED_DIR / 'manufacturer_clean.csv')
mfr.columns = [c.replace(' ', '_') for c in mfr.columns]
f.write(generate_inserts(mfr, 'manufacturers', ['manufacturer_id', 'manufacturer_name', 'slug', 'generics_count', 'brand_names_count']))


18650

In [6]:
# 2. Dosage Forms
dos = pd.read_csv(CLEANED_DIR / 'dosage_form_clean.csv')
dos.columns = [c.replace(' ', '_') for c in dos.columns]
f.write(generate_inserts(dos, 'dosage_forms', ['dosage_form_id', 'dosage_form_name', 'slug', 'brand_names_count']))


6298

In [7]:
# 3. Drug Classes
dcl = pd.read_csv(CLEANED_DIR / 'drug_class_clean.csv')
dcl_dedup = dcl.loc[dcl.groupby('drug class name')['drug class id'].idxmin()].copy()
dcl_dedup.columns = [c.replace(' ', '_') for c in dcl_dedup.columns]
f.write(generate_inserts(dcl_dedup, 'drug_classes', ['drug_class_id', 'drug_class_name', 'slug', 'generics_count']))


34260

In [8]:
# 4. Indications
ind = pd.read_csv(CLEANED_DIR / 'indication_clean.csv')
ind.columns = [c.replace(' ', '_') for c in ind.columns]
f.write(generate_inserts(ind, 'indications', ['indication_id', 'indication_name', 'slug', 'generics_count']))


147130

In [9]:
# 5. Generics
gen = pd.read_csv(CLEANED_DIR / 'generic_clean.csv')
dcl_map = pd.read_csv(MAPPINGS_DIR / 'drug_class_mapping.csv')
ind_map = pd.read_csv(MAPPINGS_DIR / 'indication_mapping.csv')

gen = trim_cols(gen)
dcl_map = trim_cols(dcl_map)
ind_map = trim_cols(ind_map)

gen = gen.merge(dcl_map, how='left', left_on='drug class', right_on='drug class name')
gen = gen.merge(ind_map, how='left', left_on='indication', right_on='indication name')

gen_cols = [
    'generic id', 'generic name', 'slug', 'monograph link',
    'drug class id', 'indication id',
    'indication description_clean', 'therapeutic class description_clean',
    'pharmacology description_clean', 'dosage description_clean',
    'administration description_clean', 'interaction description_clean',
    'contraindications description_clean', 'side effects description_clean',
    'pregnancy and lactation description_clean', 'precautions description_clean',
    'pediatric usage description_clean', 'overdose effects description_clean',
    'duration of treatment description_clean', 'reconstitution description_clean',
    'storage conditions description_clean'
]
sql_cols = [c.replace(' ', '_') for c in gen_cols]

rename_dict = dict(zip(gen_cols, sql_cols))
for old, new in rename_dict.items():
    if old in gen.columns:
        gen.rename(columns={old: new}, inplace=True)
        
f.write(generate_inserts(gen, 'generics', sql_cols))


C:\Users\ASUS-PC\AppData\Local\Temp\ipykernel_19856\35658911.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\ASUS-PC\AppData\Local\Temp\ipykernel_19856\35658911.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

7336609

In [10]:
# 6. Medicines
med = pd.read_csv(CLEANED_DIR / 'medicine_clean.csv')
gen_map = pd.read_csv(MAPPINGS_DIR / 'generic_mapping.csv')
mfr_map = pd.read_csv(MAPPINGS_DIR / 'manufacturer_mapping.csv')
dos_map = pd.read_csv(MAPPINGS_DIR / 'dosage_form_mapping.csv')

med = trim_cols(med)
gen_map = trim_cols(gen_map)
mfr_map = trim_cols(mfr_map)
dos_map = trim_cols(dos_map)

med = med.merge(gen_map, how='left', left_on='generic', right_on='generic name')
med = med.merge(mfr_map, how='left', left_on='manufacturer', right_on='manufacturer name')
med = med.merge(dos_map, how='left', left_on='dosage form', right_on='dosage form name')

med_cols = [
    'brand id', 'brand name', 'type', 'slug',
    'generic id', 'manufacturer id', 'dosage form id',
    'strength', 'strength_normalized', 'package_description',
    'price', 'pack_price', 'pack_size', 'currency'
]
sql_med_cols = [c.replace(' ', '_') for c in med_cols]

rename_dict = dict(zip(med_cols, sql_med_cols))
for old, new in rename_dict.items():
    if old in med.columns:
        med.rename(columns={old: new}, inplace=True)
        
med['currency'] = med['currency'].fillna('BDT')

f.write(generate_inserts(med, 'medicines', sql_med_cols))


C:\Users\ASUS-PC\AppData\Local\Temp\ipykernel_19856\35658911.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
C:\Users\ASUS-PC\AppData\Local\Temp\ipykernel_19856\35658911.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

3787715

In [11]:
f.close()
print("SQL insert script generated at sql/04_load_data.sql")


SQL insert script generated at sql/04_load_data.sql
